In [1]:
import sqlite3
import pandas as pd
import json
import glob
import pickle
import os
import lzma

In [2]:
# Creates all required tables including keys and constraints
def create_database(dbpath):
    conn = sqlite3.connect(dbpath)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS artist (
            id varchar(30) NOT NULL PRIMARY KEY,
            name varchar(45) DEFAULT NULL,
            genre varchar(45) DEFAULT NULL,
            popularity int DEFAULT NULL,
            followers int DEFAULT NULL
          );
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS songs (
            id varchar(30) NOT NULL PRIMARY KEY,
            name varchar(45) DEFAULT NULL,
            artist_id varchar(30) DEFAULT NULL,
            album longtext,
            CONSTRAINT fk_artist
            FOREIGN KEY (artist_id)
            REFERENCES artist(id)
        );
    ''')


    cursor.execute('''
        CREATE TABLE IF NOT EXISTS listening (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            song_id TEXT,
            ts TEXT,
            skipped INTEGER,
            FOREIGN KEY (song_id) REFERENCES songs(id)
        );
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS favorites (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            song_id TEXT,
            FOREIGN KEY (song_id) REFERENCES songs(id)
        );
    ''')


    cursor.execute('''
        CREATE TABLE IF NOT EXISTS mood (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            genre TEXT
        );
    ''')

    conn.commit()
    conn.close()

In [3]:
# Inserts data from json files into the "listening" table
def insert_listening_data(dbpath):
    conn = sqlite3.connect(dbpath)
    cursor = conn.cursor()

    json_files = glob.glob("*.json")

    dfs = []
    for file in json_files:
        with open(file, 'r') as f:
            data = json.load(f)
            df = pd.DataFrame(data)
            dfs.append(df)

    full_df = pd.concat(dfs, ignore_index=True)
    df = full_df.sort_values(by='ts')

    listening_rows = df[['spotify_track_uri', 'ts', 'skipped']].values.tolist()

    cursor.executemany("""
    INSERT INTO listening (song_id, ts, skipped)
    VALUES (?, ?, ?)
    """, listening_rows)

    conn.commit()
    conn.close()
    return df

In [4]:
# Gets extra information from cached files from the Spotify API
# Extra note - the Spotify API is only accessible to users with a Premium subscription (I do not have that). I am using cached files I created before that restriction was in place.
def gather_artist_song_info(df):
    with lzma.open("spotify_track_features.pkl.xz", "rb") as f:
        track_features = pickle.load(f)

    df_tracks = pd.DataFrame(track_features)
    df_tracks.rename(columns={"uri": "spotify_track_uri"}, inplace=True)
    df_tracks_deduped = df_tracks.drop_duplicates(subset='spotify_track_uri')
    df_merged = df.merge(df_tracks_deduped, on="spotify_track_uri", how="left")

    df_merged['artist_id'] = df_merged['artists'].apply(lambda x: x[0].get("id") if isinstance(x, list) and len(x) > 0 else None)

    with lzma.open("spotify_artists.pkl.xz", "rb") as f:
        artist_info = pickle.load(f)

    df_artists = pd.DataFrame(artist_info)
    df_artists['artist_id'] = df_artists['id']
    df_artists_deduped = df_artists.drop_duplicates(subset='artist_id')
    df_merged_2 = df_merged.merge(df_artists_deduped, on="artist_id", how="left", suffixes=("", "_artist"))

    return df_merged_2

In [5]:
# Clean and insert data into "artist" table
def insert_artist(df, dbpath):
    conn = sqlite3.connect(dbpath)
    cursor = conn.cursor()

    artist_df = (
        df[["artist_id", "name_artist", "genres", "popularity_artist", "followers"]]
        .drop_duplicates(subset="artist_id")
        .copy()
    )

    artist_df["genres"] = artist_df["genres"].apply(lambda x: ", ".join(x) if isinstance(x, list) else None)
    artist_df["followers"] = artist_df["followers"].apply(lambda x: x["total"] if isinstance(x, dict) else None)

    artist_df.rename(columns={
        "artist_id": "id",
        "name_artist": "name",
        "genres": "genre",
        "popularity_artist": "popularity"
    }, inplace=True)

    artist_rows = list(artist_df.itertuples(index=False, name=None))

    cursor.executemany("""
        INSERT OR IGNORE INTO artist
        (id, name, genre, popularity, followers)
        VALUES (?, ?, ?, ?, ?)
    """, artist_rows)

    conn.commit()
    conn.close()

In [6]:
# Clean and insert data into "songs" table
def insert_song_info(df, dbpath):
    conn = sqlite3.connect(dbpath)
    cursor = conn.cursor()

    songs_df = (
        df[["id", "name", "artist_id", "album"]]
        .dropna(subset=["id"])
        .drop_duplicates(subset="id")
        .copy()
    )

    songs_df["album"] = songs_df["album"].apply(lambda x: x["name"] if isinstance(x, dict) else x)
    song_rows = list(songs_df.itertuples(index=False, name=None))

    cursor.executemany("""
        INSERT OR IGNORE INTO songs
        (id, name, artist_id, album)
        VALUES (?, ?, ?, ?)
    """, song_rows)

    conn.commit()
    conn.close()

In [7]:
# Implement data insertion
dbpath = 'finalDB.sqlite'
if os.path.exists(dbpath):
    os.remove(dbpath)
create_database(dbpath)

df_1 = insert_listening_data(dbpath)
df_2 = gather_artist_song_info(df_1)
insert_artist(df_2, dbpath)
insert_song_info(df_2, dbpath)


In [49]:
# View data
conn = sqlite3.connect(dbpath)
artist = pd.read_sql_query("SELECT * FROM artist", conn)
conn.close()
artist

,id,name,genre,popularity,followers
0,1lMaDSraU3oiNUsVWJLHdF,Jars Of Clay,"ccm, christian rock, christian alternative roc...",46,302025
1,6BXionV4R0BunrFpSwIMUK,Britt Nicole,"christian pop, christian, cedm, ccm, christian...",51,372675
2,0TgNiaeQaWssaH9aWjbqnA,Jordan Feliz,"christian, christian pop, ccm, worship, cedm",58,446077
3,4AapPt7H6bGH4i7chTulpI,Ben Rector,,62,394816
4,26AHtbjWKiwYzsoGoUZq53,Pentatonix,christmas,64,3965254
...,...,...,...,...,...
8278,1b3LQfWsgZxMeJOiuUriP2,Nebulae Waves,jazz house,46,1207
8279,0vTVU0KH0CVzijsoKGsTPl,Barry Can't Swim,jazz house,67,252034
8280,3VsUqx5ghgNsfFQIq5LVhe,Matoux,jazz house,26,899
8281,3h9jrx2NF7x7EkNDZAn2De,Charlie Jeer,jazz house,59,31989


In [46]:
conn = sqlite3.connect(dbpath)
songs = pd.read_sql_query("SELECT * FROM songs", conn)
conn.close()
songs

,id,name,artist_id,album
0,7GEdOnkHCHDSYsFGCgczZE,God Be Merciful To Me,1lMaDSraU3oiNUsVWJLHdF,Redemption Songs
1,6wnvzxmFM1EgqOMHptPB0P,I Need Thee Every Hour,1lMaDSraU3oiNUsVWJLHdF,Redemption Songs
2,0VCp6tTtxl66A4mdy9rU18,Feel The Light,6BXionV4R0BunrFpSwIMUK,The Lost Get Found
3,6B9BxF6AAMnR86yiXhSguy,Better,6BXionV4R0BunrFpSwIMUK,Britt Nicole (Deluxe Edition)
4,1yYPjmbmac2S3PwwXPbDg6,All Day,6BXionV4R0BunrFpSwIMUK,Britt Nicole (Deluxe Edition)
...,...,...,...,...
22460,7nkbHBoyWZwWXRqYJvGyry,Shake It Off (MINGYU Solo),7nqOGRxlXj7N2JYbgNEjYH,SEVENTEEN 5th Album 'HAPPY BURSTDAY'
22461,7AQ53o110Mhvt1a2mtGP6V,Happy Virus (DK Solo),7nqOGRxlXj7N2JYbgNEjYH,SEVENTEEN 5th Album 'HAPPY BURSTDAY'
22462,0z2DWFjnHhuWjlMFqAH4GN,Destiny (WOOZI Solo),7nqOGRxlXj7N2JYbgNEjYH,SEVENTEEN 5th Album 'HAPPY BURSTDAY'
22463,2kA2Vtg4tze7xJnlsOPfmq,Gemini (JUN Solo),7nqOGRxlXj7N2JYbgNEjYH,SEVENTEEN 5th Album 'HAPPY BURSTDAY'


In [50]:
conn = sqlite3.connect(dbpath)
listening = pd.read_sql_query("SELECT * FROM listening", conn)
conn.close()
listening

,id,song_id,ts,skipped
0,1,spotify:track:7GEdOnkHCHDSYsFGCgczZE,2017-01-08T18:45:02Z,0
1,2,spotify:track:6wnvzxmFM1EgqOMHptPB0P,2017-01-08T18:45:16Z,0
2,3,spotify:track:7GEdOnkHCHDSYsFGCgczZE,2017-01-08T18:46:39Z,0
3,4,spotify:track:0VCp6tTtxl66A4mdy9rU18,2017-01-08T21:04:04Z,0
4,5,spotify:track:6B9BxF6AAMnR86yiXhSguy,2017-01-08T21:07:21Z,0
...,...,...,...,...
86782,86783,spotify:track:1lh6n7jDkg2JYyYDmbT4RS,2025-05-29T01:39:55Z,0
86783,86784,spotify:track:4sJqthsQcuyjhYbJS1JExL,2025-05-29T01:44:06Z,0
86784,86785,spotify:track:0EiLLYDKN28tRhTy0Jka5c,2025-05-29T01:44:21Z,1
86785,86786,spotify:track:1OF96rbmy2GScyaEXuXo43,2025-05-29T01:47:04Z,0


The genres and favorites tables will be populated based on user input, so I am not going to insert data until I create the front end.

In [8]:
conn = sqlite3.connect(dbpath)
listening = pd.read_sql_query("""
SELECT c.genre
FROM listening a
LEFT JOIN songs b ON CONCAT('spotify:track:', b.id) = a.song_id
LEFT JOIN artist c ON b.artist_id = c.id
WHERE c.genre IS NOT NULL
""", conn)
conn.close()
listening

,genre
0,"ccm, christian rock, christian alternative roc..."
1,"ccm, christian rock, christian alternative roc..."
2,"ccm, christian rock, christian alternative roc..."
3,"christian pop, christian, cedm, ccm, christian..."
4,"christian pop, christian, cedm, ccm, christian..."
...,...
86650,"k-pop, noise music"
86651,"k-pop, noise music"
86652,"k-pop, noise music"
86653,k-pop


In [29]:
conn = sqlite3.connect(dbpath)
listening = pd.read_sql_query("""
        SELECT COUNT(a.id), b.name, c.name
        FROM listening a
        LEFT JOIN songs b ON CONCAT('spotify:track:', b.id) = a.song_id
        LEFT JOIN artist c ON b.artist_id = c.id
        WHERE a.ts >= datetime('2025-05-29T01:49:40Z', '-1 month')
        GROUP BY b.name, c.name
        ORDER BY COUNT(a.id) DESC
        LIMIT 10
""", conn)
conn.close()
listening

,COUNT(a.id),name,name
0,23,Murmur,P1Harmony
1,22,Love Language,TOMORROW X TOGETHER
2,21,Flashy,P1Harmony
3,21,Work,P1Harmony
4,20,DUH!,P1Harmony
5,19,Pretty Boy,P1Harmony
6,18,123-78,BOYNEXTDOOR
7,18,Over And Over,P1Harmony
8,17,I Feel Good,BOYNEXTDOOR
9,15,See U Tonight (feat. YUNAH & MINJU of ILLIT),Kylie Cantrall


In [12]:
conn = sqlite3.connect(dbpath)
genre_list = pd.read_sql("""
SELECT c.genre
FROM listening a
LEFT JOIN songs b ON CONCAT('spotify:track:', b.id) = a.song_id
LEFT JOIN artist c ON b.artist_id = c.id
WHERE c.genre IS NOT NULL AND c.genre IS NOT ''
""", conn)

conn.close()


genre_counts = (
    genre_list["genre"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
    .reset_index()
)

genre_counts.columns = ["genre", "count"]
genre_counts

,genre,count
0,k-pop,40196
1,noise music,9098
2,folk pop,3271
3,christian,3001
4,christian alternative rock,2089
5,christian folk,2015
6,worship,1938
7,j-pop,1741
8,classical,1680
9,christmas,1654
